### Bitcoin

#### DLinear

In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.common import space

df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/bitcoin/bitcoin_price_news_gpt52_concise_summary.csv")
df = df.sort_values("Date").reset_index(drop=True)[['Date', 'Open',	'High',	'Low',	'Close','Volume']]	

# --- Prepare ordered data ---
df = df.sort_values("Date").reset_index(drop=True)
df_1000 = df.iloc[:1000].copy()
df_1000["item_id"] = 1

train_df = df_1000.iloc[:700]
val_df   = df_1000.iloc[700:850]
test_df  = df_1000.iloc[850:1000]

train_data = TimeSeriesDataFrame.from_data_frame(train_df, id_column="item_id", timestamp_column="Date")
val_data   = TimeSeriesDataFrame.from_data_frame(val_df,   id_column="item_id", timestamp_column="Date")
test_data  = TimeSeriesDataFrame.from_data_frame(test_df,  id_column="item_id", timestamp_column="Date")

# --- Step 1: HPO on train + val ---
predictor = TimeSeriesPredictor(
    target="Close",
    prediction_length=1,
    eval_metric="MAE",
)

predictor.fit(
    train_data,
    tuning_data=val_data,
    hyperparameters={
        "DLinear": {
            "context_length": space.Int(2, 20),
            "hidden_dimension": space.Int(8, 64),
            "kernel_size": space.Int(3, 25),
            "lr": space.Real(1e-4, 3e-3, log=True),
            "batch_size": space.Categorical(32, 64, 128),
            "max_epochs": space.Int(20, 200),
        }
    },
    hyperparameter_tune_kwargs={
        "num_trials": 20,
        "searcher": "random",
        "scheduler": "local",
    },
    enable_ensemble=False,
)

# Grab best HP
summary = predictor.fit_summary()
best_model = summary["model_best"]
best_hps = summary["model_hyperparams"][best_model]

print("Best model:", best_model)
print("Best hps:", best_hps)

# --- Step 2: walk-forward refit on train+val then predict each test point ---
# Build full history up to start of test
history_df = df_1000.iloc[:850].copy()

preds = []
for i in range(850, 1000):
    # Train data up to the point we want to predict (inclusive)
    hist_df = df_1000.iloc[:i].copy()  # up to i-1 index (1..i in 1-based)
    hist_df["item_id"] = 1
    hist_ts = TimeSeriesDataFrame.from_data_frame(hist_df, id_column="item_id", timestamp_column="Date")

    # Refit with best hyperparams
    wf_predictor = TimeSeriesPredictor(
        target="Close",
        prediction_length=1,
        eval_metric="MAE",
        path=f"ag_walkforward_{i}",  # optional; keep separate model dirs
    )
    wf_predictor.fit(
        hist_ts,
        hyperparameters={"DLinear": best_hps},
        enable_ensemble=False,
    )

    # Predict next point (i+1 in 1-based, i index in 0-based)
    fcst = wf_predictor.predict(hist_ts)
    pred_value = fcst["mean"].iloc[-1]
    true_value = df_1000.loc[i, "Close"]

    preds.append({"index": i, "date": df_1000.loc[i, "Date"], "y_true": true_value, "y_pred": pred_value})

preds_df = pd.DataFrame(preds)
print(preds_df.head())



In [ ]:
import numpy as np

err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)

#### Chronos

In [ ]:
# chronos bitcoin
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/bitcoin/bitcoin_price_news_gpt52_concise_summary.csv")
df = df.sort_values("Date").reset_index(drop=True)[['Date', 'Open',	'High',	'Low',	'Close','Volume']]	

# df_1000 already sorted by Date, item_id set, and has Close column
df_1000 = df.sort_values("Date").reset_index(drop=True).iloc[:1000].copy()
df_1000["item_id"] = 1

# Chronos predictor (no HPO needed for zero-shot)
predictor = TimeSeriesPredictor(
    target="Close",
    prediction_length=1,
)
predictor.fit(
    TimeSeriesDataFrame.from_data_frame(
        df_1000.iloc[:700], id_column="item_id", timestamp_column="Date"
    ),
    hyperparameters={
        "Chronos2": {"model_path": "autogluon/chronos-2"}
    },
    enable_ensemble=False,
)

# Rolling window prediction on test indices 850..999 (1-based 851..1000)
preds = []
for i in range(850, 1000):
    hist_df = df_1000.iloc[:i].copy()  # context 1..i (1-based)
    hist_ts = TimeSeriesDataFrame.from_data_frame(
        hist_df, id_column="item_id", timestamp_column="Date"
    )

    fcst = predictor.predict(hist_ts)
    y_pred = fcst["mean"].iloc[-1]
    y_true = df_1000.loc[i, "Close"]

    preds.append({"index": i, "date": df_1000.loc[i, "Date"], "y_true": y_true, "y_pred": y_pred})

preds_df = pd.DataFrame(preds)


In [ ]:
import numpy as np

err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)

### Energy

#### DLinear

In [ ]:
# energy
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.common import space

# Load energy data
df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/energy/Energy_with_text_summarized.csv")

# Keep only date + numeric columns (drop text)
df = df.sort_values("date").reset_index(drop=True)
numeric_cols = df.select_dtypes(include=["number"]).columns
df = df[["date"] + list(numeric_cols)]

target_col = "OT"  # change if you want a different target

# Slice first 1000 points and make single series
df_1000 = df.iloc[:1000].copy()
df_1000["item_id"] = 1

train_df = df_1000.iloc[:700]
val_df   = df_1000.iloc[700:850]
test_df  = df_1000.iloc[850:1000]

train_data = TimeSeriesDataFrame.from_data_frame(train_df, id_column="item_id", timestamp_column="date")
val_data   = TimeSeriesDataFrame.from_data_frame(val_df,   id_column="item_id", timestamp_column="date")
test_data  = TimeSeriesDataFrame.from_data_frame(test_df,  id_column="item_id", timestamp_column="date")

# --- Step 1: HPO on train + val ---
predictor = TimeSeriesPredictor(
    target=target_col,
    prediction_length=1,
    eval_metric="MAE",
)

predictor.fit(
    train_data,
    tuning_data=val_data,
    hyperparameters={
        "DLinear": {
            "context_length": space.Int(2, 20),
            "hidden_dimension": space.Int(8, 64),
            "kernel_size": space.Int(3, 25),
            "lr": space.Real(1e-4, 3e-3, log=True),
            "batch_size": space.Categorical(32, 64, 128),
            "max_epochs": space.Int(20, 200),
        }
    },
    hyperparameter_tune_kwargs={
        "num_trials": 20,
        "searcher": "random",
        "scheduler": "local",
    },
    enable_ensemble=False,
)

summary = predictor.fit_summary()
best_model = summary["model_best"]
best_hps = summary["model_hyperparams"][best_model]
print("Best model:", best_model)
print("Best hps:", best_hps)

# --- Step 2: walk-forward refit + predict ---
preds = []
for i in range(850, 1000):
    hist_df = df_1000.iloc[:i].copy()
    hist_df["item_id"] = 1
    hist_ts = TimeSeriesDataFrame.from_data_frame(hist_df, id_column="item_id", timestamp_column="date")

    wf_predictor = TimeSeriesPredictor(
        target=target_col,
        prediction_length=1,
        eval_metric="MAE",
        path=f"ag_walkforward_energy_{i}",
    )
    wf_predictor.fit(
        hist_ts,
        hyperparameters={"DLinear": best_hps},
        enable_ensemble=False,
    )

    fcst = wf_predictor.predict(hist_ts)
    y_pred = fcst["mean"].iloc[-1]
    y_true = df_1000.loc[i, target_col]

    preds.append({"index": i, "date": df_1000.loc[i, "date"], "y_true": y_true, "y_pred": y_pred})

preds_df = pd.DataFrame(preds)



#### Chronos

In [ ]:
import numpy as np

err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)


In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

df = pd.read_csv("/home/yuyan/tabdpt_mz/data/source_data/energy/Energy_with_text_summarized.csv")
df = df.sort_values("date").reset_index(drop=True)

# keep date + numeric columns (drop text)
numeric_cols = df.select_dtypes(include=["number"]).columns
df = df[["date"] + list(numeric_cols)]

target_col = "OT"

df_1000 = df.iloc[:1000].copy()
df_1000["item_id"] = 1

# fit once on the initial train segment (1..700)
predictor = TimeSeriesPredictor(
    target=target_col,
    prediction_length=1,
)

predictor.fit(
    TimeSeriesDataFrame.from_data_frame(
        df_1000.iloc[:700], id_column="item_id", timestamp_column="date"
    ),
    hyperparameters={        "Chronos2": {"model_path": "autogluon/chronos-2"}
},  # adjust if you use Chronos-2 name
    enable_ensemble=False,
)

# rolling-context predictions for 851..1000
preds = []
for i in range(850, 1000):
    hist_df = df_1000.iloc[:i].copy()
    hist_ts = TimeSeriesDataFrame.from_data_frame(
        hist_df, id_column="item_id", timestamp_column="date"
    )

    fcst = predictor.predict(hist_ts)
    y_pred = fcst["mean"].iloc[-1]
    y_true = df_1000.loc[i, target_col]

    preds.append({"index": i, "date": df_1000.loc[i, "date"], "y_true": y_true, "y_pred": y_pred})

preds_df = pd.DataFrame(preds)
# mae = (preds_df["y_true"] - preds_df["y_pred"]).abs().mean()
# print("Chronos rolling MAE (energy):", mae)


In [ ]:
import numpy as np
err = preds_df["y_true"] - preds_df["y_pred"]

mae = err.abs().mean()
rmse = np.sqrt(np.mean(err ** 2))

# Avoid divide-by-zero in MAPE
eps = 1e-8
mape = np.mean(np.abs(err) / (np.abs(preds_df["y_true"]) + eps)) * 100

print("MAE:", mae)

print("RMSE:", rmse)
print("MAPE (%):", mape)